# ECHO Sheet Maker

This notebook turns an experiment (a grid of reagents × runs) into a Beckman **ECHO transfer sheet**, plus helper tabs and plate-map images, all written back into your Excel file.

## How to use it
1. **Prepare your Excel file** with two tabs: `experiment_grid` and `reagent_concentrations` (see structure below).
2. **Open the CONFIGURATION cell** (the first code cell) and set:
   - `file_path` &rarr; the path to your Excel file
   - `PLATE_TYPE` &rarr; **96, 384, or 1536** (this is the *destination* plate)
3. **Run every cell top to bottom.** You will be asked to type a starting well **twice** (source, then destination).
4. Open the Excel file when finished. The tab you deliver to the ECHO software is **`ECHO_SHEET`**.

---

## Choosing your plate type (96 / 384 / 1536)

There is now **one** place to switch plate types: the `PLATE_TYPE` line in the CONFIGURATION cell.

```python
PLATE_TYPE = 96     # options: 96, 384, 1536
```

| PLATE_TYPE | Destination layout | Default destination name |
|-----------|--------------------|--------------------------|
| `96`      | 8 rows × 12 cols   | `96_Dest`                |
| `384`     | 16 rows × 24 cols  | `384PP_Dest`             |
| `1536`    | 32 rows × 48 cols  | `1536LDV_Dest`           |

The **source** plate is always a **384-well** Echo source plate (16 × 24) regardless of the destination — that is a hardware fact of the ECHO, so you never change it.

---

## Source-well volume limits (dead volume vs. usable volume)

Two numbers control how much liquid goes into each **source** well. They are defined at the top of the CONFIGURATION cell so you never have to hunt for them:

```python
MAX_SOURCE_WELL_VOLUME_NL  = 63000   # hard cap: the MOST you may put in ANY source well
SOURCE_WELL_DEAD_VOLUME_NL = 18000   # dead volume the ECHO physically cannot aspirate
```

- **Dead volume** is the liquid that stays trapped at the bottom of the well and can never be dispensed. Every source well is loaded with this extra amount on top of what it dispenses.
- **Usable volume** = `MAX_SOURCE_WELL_VOLUME_NL − SOURCE_WELL_DEAD_VOLUME_NL`. A reagent that needs to dispense more than this is automatically **split across multiple source wells** (e.g. `Water_1`, `Water_2`).
- The notebook **checks** every source well and warns you if any well would exceed `MAX_SOURCE_WELL_VOLUME_NL`.

So: to change the maximum you can load into a source well, edit **`MAX_SOURCE_WELL_VOLUME_NL`** only.

---

### Example `experiment_grid` tab

| Run | CFPS_MM | DNA_Lime | Pyruvate(25mM) | HEPES | Water |
|-----|---------|----------|----------------|-------|-------|
| 1   | 675     | 200      | 0              | 100   | 125   |
| 2   | 675     | 200      | 25             | 100   | 100   |

### Example `reagent_concentrations` tab

| Reagent        | Molarity | Unit  |
|----------------|----------|-------|
| CFPS_MM        | 100      | mg/mL |
| DNA_Lime       | 100      | mg/mL |
| Pyruvate(25mM) | 25       | mM    |
| HEPES          | 1000     | mM    |
| Water          |          |       |

**Notes**
- Each run is one well of the destination plate. Volumes are in **nanoliters**, in 25 nL increments.
- Reagent column names in `experiment_grid` must match `reagent_concentrations` **exactly**.
- Blank cells are treated as 0. Molarity/Unit may be blank (e.g. Water).


In [ ]:
# ===================== CONFIGURATION =====================
import os
import re
import tempfile
import numpy as np
import pandas as pd

# --- 1. Path to your Excel file -------------------------------------------
file_path = r'C:\Users\d_gar\research\code_and_scripts\Salt_calibration_3_9_26.xlsx'

# --- 2. DESTINATION plate type: 96, 384, or 1536 --------------------------
PLATE_TYPE = 96          # <-- change this one line to switch plates

# --- 3. Source-well volume limits (nL) ------------------------------------
MAX_SOURCE_WELL_VOLUME_NL  = 63000   # hard cap: most you may load into ANY source well
SOURCE_WELL_DEAD_VOLUME_NL = 18000   # dead volume the ECHO cannot aspirate
# Usable volume that can actually be dispensed from one source well:
USABLE_VOLUME_PER_WELL = MAX_SOURCE_WELL_VOLUME_NL - SOURCE_WELL_DEAD_VOLUME_NL

# --- 4. Plate geometry (source is ALWAYS a 384 Echo plate) ----------------
PLATE_CONFIG = {
    96:   {'rows': 8,  'cols': 12, 'dest_name': '96_Dest'},
    384:  {'rows': 16, 'cols': 24, 'dest_name': '384PP_Dest'},
    1536: {'rows': 32, 'cols': 48, 'dest_name': '1536LDV_Dest'},
}
SOURCE_ROWS, SOURCE_COLS = 16, 24     # 384-well source, fixed

# --- 5. Sheet names -------------------------------------------------------
experiment_grid_sheet_name = 'experiment_grid'
dispensing_sheet_name      = 'dispensing_volume'
conditions_sheet_name      = 'dispensing_conditions'

# ==========================================================================
# Validation
if PLATE_TYPE not in PLATE_CONFIG:
    raise ValueError(f"PLATE_TYPE must be one of {list(PLATE_CONFIG)}, got {PLATE_TYPE}.")

DEST_ROWS = PLATE_CONFIG[PLATE_TYPE]['rows']
DEST_COLS = PLATE_CONFIG[PLATE_TYPE]['cols']
DEST_PLATE_NAME = PLATE_CONFIG[PLATE_TYPE]['dest_name']

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"File not found at {file_path}\n"
        "Update the file_path variable with the correct path to your Excel file."
    )

print(f"File found:        {file_path}")
print(f"Destination plate: {PLATE_TYPE}-well ({DEST_ROWS} rows x {DEST_COLS} cols)")
print(f"Source plate:      384-well (fixed)")
print(f"Max per source well: {MAX_SOURCE_WELL_VOLUME_NL} nL "
      f"(dead {SOURCE_WELL_DEAD_VOLUME_NL} nL, usable {USABLE_VOLUME_PER_WELL} nL)")

df_preview = pd.read_excel(file_path, sheet_name=experiment_grid_sheet_name)
print("\nSuccessfully loaded experiment_grid:")
display(df_preview.head())


In [ ]:
# ===================== SHARED HELPER FUNCTIONS =====================
# Defined once here so every later cell uses the same, consistent logic.

def generate_plate_rows(n_rows):
    """Row labels A..Z, then AA..AF for plates taller than 26 rows."""
    rows = []
    for i in range(n_rows):
        rows.append(chr(ord('A') + i) if i < 26 else 'A' + chr(ord('A') + i - 26))
    return rows

def parse_well(well):
    """'AB12' -> ('AB', 12). Robust for one- or two-letter row labels."""
    well = str(well).strip().upper()
    i = 0
    while i < len(well) and well[i].isalpha():
        i += 1
    if i == 0 or i == len(well):
        raise ValueError(f"Invalid well address: '{well}'")
    return well[:i], int(well[i:])

def well_list(n_rows, n_cols):
    """All wells in row-major order for a plate of the given size."""
    rows = generate_plate_rows(n_rows)
    return [f"{r}{c}" for r in rows for c in range(1, n_cols + 1)]

def well_to_grid_coords(well, n_rows, n_cols):
    """Return (row_index, col_index) for a well on an n_rows x n_cols plate."""
    row_label, col_num = parse_well(well)
    rows = generate_plate_rows(n_rows)
    if row_label not in rows:
        raise ValueError(f"Row '{row_label}' out of range for a {n_rows}-row plate.")
    if not (1 <= col_num <= n_cols):
        raise ValueError(f"Column {col_num} out of range for an {n_cols}-column plate.")
    return rows.index(row_label), col_num - 1

def rotate_well_pattern(start_well, n_rows, n_cols):
    """Row-major well list starting at start_well and wrapping around."""
    pattern = well_list(n_rows, n_cols)
    start_well = str(start_well).strip().upper()
    if start_well not in pattern:
        raise ValueError(
            f"Invalid starting well '{start_well}' for a {n_rows}x{n_cols} plate."
        )
    idx = pattern.index(start_well)
    return pattern[idx:] + pattern[:idx]

print("Helper functions loaded.")


In [ ]:
# ===================== STEP 1: split volumes across source wells =====================
# Any reagent needing more than USABLE_VOLUME_PER_WELL is split into _1, _2, ...
df_experiment_grid = pd.read_excel(file_path, sheet_name=experiment_grid_sheet_name)

new_df = pd.DataFrame(df_experiment_grid['Run'])

def add_new_column(df, col, suffix):
    new_col = f"{col}_{suffix}"
    df[new_col] = 0
    return new_col

for col in df_experiment_grid.columns:
    if col == 'Run':
        continue
    suffix = 1
    new_col = add_new_column(new_df, col, suffix)
    running_total = 0
    for i, value in enumerate(df_experiment_grid[col]):
        value = 0 if pd.isna(value) else value
        if running_total + value > USABLE_VOLUME_PER_WELL:
            suffix += 1
            new_col = add_new_column(new_df, col, suffix)
            running_total = 0
        new_df.at[i, new_col] = value
        running_total += value

with pd.ExcelWriter(file_path, mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
    new_df.to_excel(writer, sheet_name=dispensing_sheet_name, index=False)

print(f"'{dispensing_sheet_name}' sheet created "
      f"(usable volume per source well = {USABLE_VOLUME_PER_WELL} nL).")
display(new_df.head())


In [ ]:
# ===================== STEP 2: assign source wells & volumes =====================
df_dispensing_volume = pd.read_excel(file_path, sheet_name=dispensing_sheet_name)

# Total volume each source column must dispense, plus the dead volume that stays behind.
total_dispensed_volumes = df_dispensing_volume.drop(columns=['Run']).sum().to_dict()

dispensing_conditions = pd.DataFrame(df_dispensing_volume.columns[1:], columns=['Sample ID'])
dispensing_conditions['Well Volume (nL)'] = (
    dispensing_conditions['Sample ID'].map(total_dispensed_volumes) + SOURCE_WELL_DEAD_VOLUME_NL
)

# --- Safety check against the hard cap -----------------------------------
over_cap = dispensing_conditions[
    dispensing_conditions['Well Volume (nL)'] > MAX_SOURCE_WELL_VOLUME_NL
]
if not over_cap.empty:
    print("WARNING: these source wells exceed the "
          f"{MAX_SOURCE_WELL_VOLUME_NL} nL cap:")
    display(over_cap)
    print("Consider lowering per-run volumes; splitting alone did not resolve it.")
else:
    print(f"All source wells are within the {MAX_SOURCE_WELL_VOLUME_NL} nL cap.")

# --- Source wells: ALWAYS on a 384 plate ---------------------------------
start_well = input("Enter the starting SOURCE well (e.g. A1), source plate is 384-well: ").strip().upper()
source_pattern = rotate_well_pattern(start_well, SOURCE_ROWS, SOURCE_COLS)
if len(dispensing_conditions) > len(source_pattern):
    raise ValueError(
        f"{len(dispensing_conditions)} source wells needed but only "
        f"{len(source_pattern)} available on a 384 plate."
    )
dispensing_conditions['Source Well'] = source_pattern[:len(dispensing_conditions)]

# --- Source plate types ---------------------------------------------------
def get_source_plate_types(sample_ids, default_type='384PP_Plus_AQ'):
    out = []
    for sid in sample_ids:
        s = str(sid).lower()
        out.append('384PP_Plus_AQ_BP' if ('extract' in s or 'cfps' in s) else default_type)
    return out

dispensing_conditions['Source Plate Type'] = get_source_plate_types(
    dispensing_conditions['Sample ID'].tolist()
)

with pd.ExcelWriter(file_path, mode='a', if_sheet_exists='replace', engine='openpyxl') as writer:
    dispensing_conditions.to_excel(writer, sheet_name=conditions_sheet_name, index=False)

print(f"\n'{conditions_sheet_name}' sheet created.")
display(dispensing_conditions.head())


In [ ]:
# ===================== STEP 3: build the ECHO_SHEET =====================
def run_to_well(run, start_well='A1'):
    """Map a run number to a destination well on the chosen plate."""
    wells = well_list(DEST_ROWS, DEST_COLS)
    r, c = parse_well(start_well)
    rows = generate_plate_rows(DEST_ROWS)
    if r not in rows or not (1 <= c <= DEST_COLS):
        raise ValueError(f"Invalid destination start well '{start_well}' for {PLATE_TYPE}-well plate.")
    start_index = rows.index(r) * DEST_COLS + (c - 1)
    return wells[(int(run) - 1 + start_index) % len(wells)]

dest_start = input(
    f"Enter the starting DESTINATION well (e.g. A1), plate is {PLATE_TYPE}-well "
    f"({DEST_ROWS}x{DEST_COLS}): "
).strip().upper()

df_source_data = pd.read_excel(file_path, sheet_name=conditions_sheet_name)

df_melted = df_dispensing_volume.melt(id_vars='Run', var_name='Sample ID', value_name='Transfer Volume')
df_melted = df_melted[df_melted['Transfer Volume'] != 0].copy()
df_melted.dropna(subset=['Transfer Volume'], inplace=True)
df_melted['Run'] = pd.to_numeric(df_melted['Run'], errors='coerce')
df_melted.dropna(subset=['Run'], inplace=True)
df_melted['Run'] = df_melted['Run'].astype(int)

df_melted['Destination Well'] = df_melted['Run'].apply(lambda x: run_to_well(x, dest_start))
df_melted = pd.merge(
    df_melted,
    df_source_data[['Sample ID', 'Source Well', 'Source Plate Type']],
    on='Sample ID', how='left'
)

df_melted['Source Plate'] = 'Source[1]'
df_melted['Destination Plate Name'] = DEST_PLATE_NAME
df_melted['Destination Well X Offset'] = 0
df_melted['Destination Well Y Offset'] = 0

df_melted = df_melted[['Source Plate', 'Source Plate Type', 'Source Well', 'Sample ID',
                       'Destination Plate Name', 'Destination Well', 'Transfer Volume',
                       'Destination Well X Offset', 'Destination Well Y Offset']]

with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
    df_melted.to_excel(writer, sheet_name='ECHO_SHEET', index=False)

print("ECHO_SHEET created.")
display(df_melted.head())


In [ ]:
# ===================== STEP 4: plate map images (source + destination) =====================
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.colors import LinearSegmentedColormap, Normalize
from openpyxl.drawing.image import Image as XLImage
from openpyxl import load_workbook


def base_reagent(sample_id):
    """'Water_2' -> 'Water'; only strips a trailing numeric suffix."""
    parts = str(sample_id).split('_')
    if len(parts) > 1 and parts[-1].isdigit():
        return '_'.join(parts[:-1])
    return str(sample_id)


def build_source_plate_figure():
    """Colour-coded 384 source plate: fill = volume, dot = reagent."""
    df_cond = pd.read_excel(file_path, sheet_name=conditions_sheet_name)
    volume_data = df_cond.set_index('Source Well')['Well Volume (nL)']

    cmap = LinearSegmentedColormap.from_list("vol", ["yellow", "orange", "red"])
    vmin = volume_data.min() if not volume_data.empty else 0
    vmax = volume_data.max() if not volume_data.empty else 1
    norm = Normalize(vmin - 1, vmax + 1) if vmin == vmax else Normalize(vmin, vmax)

    reagents = df_cond['Sample ID'].apply(base_reagent).unique()
    palette = ['blue', 'green', 'purple', 'pink', 'orange', 'gray', 'brown', 'red',
               'cyan', 'magenta', 'lime', 'teal', 'navy', 'maroon', 'olive', 'indigo']
    reagent_colors = {r: palette[i % len(palette)] for i, r in enumerate(reagents)}

    fig, ax = plt.subplots(figsize=(SOURCE_COLS * 0.7, SOURCE_ROWS * 0.7))
    for well, vol in volume_data.items():
        try:
            ri, ci = well_to_grid_coords(well, SOURCE_ROWS, SOURCE_COLS)
        except ValueError as e:
            print(f"Skipping source well {well}: {e}")
            continue
        ax.add_patch(patches.Rectangle((ci, ri), 1, 1, facecolor=cmap(norm(vol)), edgecolor='none'))
        sid = df_cond.loc[df_cond['Source Well'] == well, 'Sample ID'].iloc[0]
        rname = base_reagent(sid)
        ax.add_patch(patches.Rectangle((ci, ri), 0.2, 0.2,
                     facecolor=reagent_colors.get(rname, 'black'), edgecolor='black', lw=0.5))
        ax.text(ci + 0.5, ri + 0.35, f'{int(vol)}', ha='center', va='center', fontsize=8, weight='bold')
        ax.text(ci + 0.5, ri + 0.7, rname, ha='center', va='center', fontsize=6)

    for ri in range(SOURCE_ROWS):
        for ci in range(SOURCE_COLS):
            ax.add_patch(patches.Rectangle((ci, ri), 1, 1, facecolor='none', edgecolor='black', lw=0.5))

    ax.set_xlim(0, SOURCE_COLS); ax.set_ylim(0, SOURCE_ROWS)
    ax.set_xticks(np.arange(0.5, SOURCE_COLS), labels=np.arange(1, SOURCE_COLS + 1))
    ax.set_yticks(np.arange(0.5, SOURCE_ROWS), labels=generate_plate_rows(SOURCE_ROWS))
    ax.set_xlabel('Column'); ax.set_ylabel('Row')
    ax.set_title('384-Well Source Plate — Volume Map (nL)')
    ax.set_aspect('equal', adjustable='box'); ax.invert_yaxis()

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm); sm.set_array([])
    fig.colorbar(sm, ax=ax, fraction=0.02, pad=0.04).set_label('Well Volume (nL)')
    handles = [patches.Patch(color=c, label=r) for r, c in reagent_colors.items()]
    ax.legend(handles=handles, bbox_to_anchor=(0.5, -0.12), loc='upper center',
              ncol=min(len(handles), 6), fancybox=True)
    fig.tight_layout(rect=[0, 0.08, 1, 1])
    return fig


def build_destination_plate_figure():
    """Stacked reagent-proportion map of the destination plate."""
    dfm = df_melted.copy()
    dfm['Base Reagent'] = dfm['Sample ID'].apply(base_reagent)
    vols = dfm.groupby(['Destination Well', 'Base Reagent'])['Transfer Volume'].sum().reset_index()
    props = vols.pivot_table(index='Destination Well', columns='Base Reagent',
                             values='Transfer Volume', fill_value=0)
    props = props.div(props.sum(axis=1), axis=0)

    reagents = [c for c in props.columns if not str(c).startswith('Unnamed:')]
    palette = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b',
               '#e377c2', '#7f7f7f', '#bcbd22', '#17becf', '#aec7e8', '#ffbb78',
               '#98df8a', '#ff9896', '#c5b0d5', '#c49c94', '#f7b6d2', '#c7c7c7',
               '#dbdb8d', '#9edae5']
    reagent_colors = {r: palette[i % len(palette)] for i, r in enumerate(reagents)}

    fig, ax = plt.subplots(figsize=(DEST_COLS * 0.5, DEST_ROWS * 0.5))
    for well, row in props.iterrows():
        try:
            ri, ci = well_to_grid_coords(well, DEST_ROWS, DEST_COLS)
        except ValueError as e:
            print(f"Skipping destination well {well}: {e}")
            continue
        vals = row[reagents]
        tot = vals.sum()
        if tot == 0:
            continue
        bottom = 0
        for r in reagents:
            p = vals.get(r, 0) / tot
            if p > 0:
                ax.add_patch(patches.Rectangle((ci, ri + bottom), 1, p,
                             facecolor=reagent_colors[r], edgecolor='none'))
                bottom += p

    for ri in range(DEST_ROWS):
        for ci in range(DEST_COLS):
            ax.add_patch(patches.Rectangle((ci, ri), 1, 1, facecolor='none', edgecolor='black', lw=0.5))

    ax.set_xlim(0, DEST_COLS); ax.set_ylim(0, DEST_ROWS)
    ax.set_xticks(np.arange(0.5, DEST_COLS), labels=np.arange(1, DEST_COLS + 1))
    ax.set_yticks(np.arange(0.5, DEST_ROWS), labels=generate_plate_rows(DEST_ROWS))
    ax.set_xlabel('Column'); ax.set_ylabel('Row')
    ax.set_title(f'{PLATE_TYPE}-Well Destination Plate — Reagent Distribution')
    ax.set_aspect('equal', adjustable='box'); ax.invert_yaxis()
    handles = [patches.Patch(color=reagent_colors[r], label=r) for r in reagents]
    ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0.)
    fig.tight_layout()
    return fig


def save_figure_to_sheet(fig, sheet_name):
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as tmp:
        img_path = tmp.name
    fig.savefig(img_path, bbox_inches='tight', dpi=200)
    wb = load_workbook(file_path)
    if sheet_name in wb.sheetnames:
        del wb[sheet_name]
    ws = wb.create_sheet(sheet_name)
    ws.add_image(XLImage(img_path), 'A1')
    wb.save(file_path)
    os.remove(img_path)


# --- Build, show, and SAVE both plate maps -------------------------------
source_fig = build_source_plate_figure()
plt.show()
save_figure_to_sheet(source_fig, 'Source Plate Map')
plt.close(source_fig)

dest_fig = build_destination_plate_figure()
plt.show()
save_figure_to_sheet(dest_fig, 'Destination Plate Map')
plt.close(dest_fig)

print("Saved both images: 'Source Plate Map' and 'Destination Plate Map' sheets.")


## Concentration labels (Headings + well headers)

The next two cells compute the final concentration of each reagent in each destination well
`(transfer_volume × stock_molarity) / total_well_volume` and write two helper tabs:
`Headings` (a reagent × well table) and `well headers` (a one-line label per well).


In [ ]:
# ===================== STEP 5: concentration table (Headings sheet) =====================
try:
    df_reagent_concentrations = pd.read_excel(file_path, sheet_name='reagent_concentrations')
    reagent_data = df_reagent_concentrations.set_index('Reagent')[['Molarity', 'Unit']].to_dict('index')
    print("Loaded reagent_concentrations.")
except Exception as e:
    print(f"Could not load reagent_concentrations: {e}\nSkipping concentration steps.")
    reagent_data = {}

if reagent_data:
    df_melted['Base Reagent'] = df_melted['Sample ID'].apply(base_reagent)

    totals = (df_melted.groupby('Destination Well')['Transfer Volume']
              .sum().reset_index().rename(columns={'Transfer Volume': 'Total_Well_Volume'}))
    df_merged = pd.merge(df_melted, totals, on='Destination Well', how='left')

    def final_conc(row):
        r = row['Base Reagent']
        info = reagent_data.get(r)
        if info and pd.notna(info.get('Molarity')) and row['Total_Well_Volume'] > 0:
            return round((row['Transfer Volume'] * info['Molarity']) / row['Total_Well_Volume'], 2)
        return 0.0

    df_merged['Final Concentration'] = df_merged.apply(final_conc, axis=1)

    agg = df_merged.groupby(['Base Reagent', 'Destination Well'])['Final Concentration'].max().reset_index()
    df_concentrations_pivot = agg.pivot_table(index='Base Reagent', columns='Destination Well',
                                              values='Final Concentration', fill_value=0)

    def sort_key(col):
        m = re.match(r'([A-Za-z]+)(\d+)', str(col))
        return (m.group(1), int(m.group(2))) if m else (str(col), 0)

    df_concentrations_pivot = df_concentrations_pivot[
        sorted(df_concentrations_pivot.columns, key=sort_key)
    ]

    with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_concentrations_pivot.to_excel(writer, sheet_name='Headings', index=True)

    print("Headings sheet created.")
    display(df_concentrations_pivot.iloc[:, :5])
else:
    print("Headings sheet not created.")


In [ ]:
# ===================== STEP 6: one-line well labels (well headers sheet) =====================
if reagent_data and 'df_concentrations_pivot' in dir():
    well_headers = {}
    for well in df_concentrations_pivot.columns:
        parts = []
        for reagent in df_concentrations_pivot.index:
            conc = df_concentrations_pivot.loc[reagent, well]
            if conc > 0:
                unit = reagent_data.get(reagent, {}).get('Unit', '')
                parts.append(f"{reagent}: {conc} {unit}".strip() if (pd.notna(unit) and unit)
                             else f"{reagent}: {conc}")
        well_headers[well] = ", ".join(parts) if parts else "Empty"

    df_well_headers = pd.DataFrame(list(well_headers.items()), columns=['Well', 'Header'])
    with pd.ExcelWriter(file_path, engine='openpyxl', mode='a', if_sheet_exists='replace') as writer:
        df_well_headers.to_excel(writer, sheet_name='well headers', index=False)

    print("well headers sheet created.")
    display(df_well_headers.head(10))
else:
    print("well headers sheet not created.")
